# 02 — Data preparation and baseline models

This notebook defines a reproducible holdout and runnable baseline evaluation code. It does not remove duplicate rows. Training, resampling, cross-validation, and hyperparameter tuning are deliberately deferred until the reader chooses to execute the corresponding cells and plan the next phase.

Start with [Notebook 01](01_dataset_understanding_eda.ipynb) so the pinned CSV is available under `data/raw/`. The holdout keeps exact-feature duplicates together; it is a learning benchmark, not evidence of production generalization.


## Load data and define inputs

A conventional `train_test_split` with `stratify=y` assigns rows randomly while approximately preserving each class proportion in both partitions. This is a useful first pattern for imbalanced classification, but it does not keep duplicate feature signatures together.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

working_directory = Path.cwd().resolve()
repository_candidates = (working_directory, *working_directory.parents)
REPO_ROOT = next(
    (candidate for candidate in repository_candidates
     if (candidate / 'README.md').is_file() and (candidate / 'notebooks').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')
RAW_CSV = REPO_ROOT / 'data' / 'raw' / 'creditcard.csv'
if not RAW_CSV.is_file():
    raise FileNotFoundError('Run Notebook 01 to retrieve the pinned dataset before continuing.')

df = pd.read_csv(RAW_CSV)
X = df.drop(columns=['Class'])
y = df['Class']
print(f'Input rows: {len(X):,}; numeric inputs: {X.shape[1]}')


Input rows: 284,807; numeric inputs: 30


## Conventional stratified split: useful, but not enough here

`stratify=y` makes each partition's class proportions approximately match the full target distribution. It does not group identical feature rows. For the fixed conventional split below, the supplied diagnostics report 227,845 training rows and 56,962 test rows, with 394 and 98 frauds respectively. They also report 293 distinct exact-feature signatures present on both sides. The rows must not be deleted to hide this overlap.


In [2]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)

# Hash X only, not the label, to identify identical feature signatures.
feature_groups = pd.util.hash_pandas_object(X, index=False).astype('uint64')
random_train_groups = feature_groups.loc[X_train_random.index].unique()
random_test_groups = feature_groups.loc[X_test_random.index].unique()
random_overlap = np.intersect1d(random_train_groups, random_test_groups)
print(f'Conventional split rows: {len(X_train_random):,} train / {len(X_test_random):,} test')
print(f'Frauds: {int(y_train_random.sum())} train / {int(y_test_random.sum())} test')
print(f'Distinct exact-feature signatures crossing the split: {len(random_overlap)}')


Conventional split rows: 227,845 train / 56,962 test
Frauds: 394 train / 98 test
Distinct exact-feature signatures crossing the split: 293


## Reproducible group-aware holdout

The structural audit found 1,081 exact duplicate rows (0.3796%) and zero feature-conflicting-target groups. Do not delete duplicates: retain the observations, but assign rows with identical input features to the same partition.

The next cell hashes **X only**, so the target cannot influence group membership, then takes the first fold from `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`. Stratification approximately preserves class proportions while grouping prevents an identical feature signature from appearing in both train and test. The supplied diagnostic result is 227,845 train / 56,962 test, with 393 / 99 frauds, fraud rates 0.1725% / 0.1738%, and zero group overlap.


In [3]:
group_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_indices, test_indices = next(
    group_splitter.split(X, y, groups=feature_groups.to_numpy())
)
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

train_groups = feature_groups.iloc[train_indices].unique()
test_groups = feature_groups.iloc[test_indices].unique()
group_overlap = np.intersect1d(train_groups, test_groups)
split_summary = pd.DataFrame({
    'rows': [len(X_train), len(X_test)],
    'frauds': [int(y_train.sum()), int(y_test.sum())],
    'fraud_rate_pct': [100 * y_train.mean(), 100 * y_test.mean()],
}, index=['train', 'test'])
print(f'Exact-feature group overlap: {len(group_overlap)}')
assert len(group_overlap) == 0
split_summary


Exact-feature group overlap: 0


,rows,frauds,fraud_rate_pct
train,227845,393,0.172486
test,56962,99,0.173800


### Split interpretation

The grouped holdout keeps all rows, including exact duplicates, but places each exact-feature signature wholly in one partition. It therefore answers a stricter question than a row-random split: how does a model behave on held-out feature signatures? A group-aware split does not solve the dataset's short historical time span or establish production performance. The observed fold counts and rates above are diagnostic reference values, not model results.


## Preprocessing and SMOTE

`Time`, `Amount`, and the anonymized PCA features are all numeric. Their raw scales differ substantially: the supplied profile reports standard deviation 1.9587 for V1 and 0.3301 for V28, while `Time` and `Amount` have much larger raw scales. Standardize **all 30 numeric inputs**, not just `Time` and `Amount`, for the MLP and especially for SMOTE's nearest-neighbor distance calculation. Never scale the target `Class`.

Trees do not intrinsically require standardized features, but SMOTE does use distances. Put `StandardScaler` and `SMOTE(random_state=42)` inside an `imblearn.pipeline.Pipeline` so the scaler is fit on each training partition and synthetic samples are generated only from that training partition. Test/validation data keep the original class prevalence and are transformed by the fitted scaler without resampling. In future cross-validation, pass this complete pipeline to the CV routine so each fold fits its own scaler and resampler.

SMOTE is declared below but is not run in this notebook during artifact creation.


## Baseline model definitions

These are intentionally simple reference classifiers. The following cell only creates estimator objects; it does not fit them.


In [4]:
models = {
    'Decision tree': DecisionTreeClassifier(random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(random_state=42),
}


## One holdout training and evaluation loop

The next cell is runnable when you choose to execute it. For each baseline it fits the scaler, applies SMOTE to training data only, and fits the classifier. It then scores the untouched, naturally imbalanced test partition. Average Precision is the PR-AUC summary; precision, recall, and F2 use each estimator's default prediction threshold. The confusion matrix uses rows = true class and columns = predicted class, ordered `[0, 1]`. Accuracy is intentionally omitted.

This is a single holdout baseline, not cross-validation or hyperparameter tuning. Both are deferred to the next phase. In particular, future CV must receive the complete imbalanced-learn pipeline so SMOTE cannot sample across a validation fold.


In [ ]:
metric_rows = []
for model_name, classifier in models.items():
    pipeline = ImbPipeline(steps=[
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', classifier),
    ])
    pipeline.fit(X_train, y_train)
    y_score = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)

    metric_rows.append({
        'model': model_name,
        'Average Precision (PR-AUC summary)': average_precision_score(y_test, y_score),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'F2': fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    })
    print(f'{model_name} confusion matrix (rows=true, columns=predicted; labels 0, 1):')
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

baseline_summary = pd.DataFrame(metric_rows).set_index('model')
baseline_summary


## Proposed MLP design — hypothesis, not a trained result

The compact network size is a starting hypothesis for 30 tabular numeric inputs and only 492 fraud observations, not a claim that this architecture is optimal. Use binary log-loss (cross-entropy), ReLU hidden units, L2 regularization through `alpha`, and the Adam optimizer. Starting settings are hidden layers `(64, 32)`, `learning_rate_init=1e-3`, `alpha=1e-4`, `batch_size=256`, `max_iter=100`, and random seed 42. These settings must be validated, not assumed optimal.

Do not enable `early_stopping` inside the post-SMOTE estimator: it would create an internal validation split after resampling rather than provide the intended natural-prevalence validation. Validation and CV design belong to the next phase. There are no categorical features, card/customer IDs, or behavioral sequences, so embeddings and sequential networks have no supported input structure here.


In [5]:
mlp_hypothesis = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    alpha=1e-4,
    batch_size=256,
    max_iter=100,
    random_state=42,
)


## What remains for the next phase

- Execute baseline training only after reviewing the holdout and the implications of the duplicate-group split.
- Design validation and cross-validation explicitly; keep all preprocessing and SMOTE inside each training fold.
- Tune decision thresholds against validation predictions at natural prevalence and report fraud recall alongside false-alert burden.
- Add the proposed MLP as a separate experiment and compare it with the baselines without treating its initial settings as optimal.
- State the two-day 2013 data limitation in every performance summary; this holdout cannot establish current, cross-issuer, or production generalization.
